In [1]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [2]:
import medmnist
from medmnist import BreastMNIST, PneumoniaMNIST
from calculate_L import *

In [3]:
# preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize(mean=[.5], std=[.5])
])

In [4]:
train_dataset = BreastMNIST(root='./data/', split='train', download=True, transform=data_transform)
test_dataset = BreastMNIST(root='./data/', split='test', download=True, transform=data_transform)

Using downloaded and verified file: ./data/breastmnist.npz
Using downloaded and verified file: ./data/breastmnist.npz


In [5]:
train_bs = len(train_dataset)
test_bs = len(test_dataset)
train_bs, test_bs

(546, 156)

In [6]:
# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=train_bs, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=test_bs, shuffle=False)

In [7]:
print(train_dataset)
print("===================")
print(test_dataset)

Dataset BreastMNIST of size 28 (breastmnist)
    Number of datapoints: 546
    Root location: ./data/
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'malignant', '1': 'normal, benign'}
    Number of samples: {'train': 546, 'val': 78, 'test': 156}
    Description: The BreastMNIST is based on a dataset of 780 breast ultrasound images. It is categorized into 3 classes: normal, benign, and malignant. As we use low-resolution images, we simplify the task into binary classification by combining normal and benign as positive and classifying them against malignant as negative. We split the source dataset with a ratio of 7:1:2 into training, validation and test set. The source images of 1×500×500 are resized into 1×28×28.
    License: CC BY 4.0
Dataset BreastMNIST of size 28 (breastmnist)
    Number of datapoints: 156
    Root location: ./data/
    Split: test
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'malignant

In [8]:
# visualization
train_dataset.montage(length=1)

In [9]:
X_test, Y_test = next(iter(test_loader))
X_test =X_test.flatten(start_dim=1).cpu().numpy().T
Y_test= Y_test.cpu().numpy()
X_test.shape,Y_test.shape

((784, 156), (156, 1))

In [10]:
X,Y = next(iter(train_loader))


In [11]:
X.shape, Y.shape

(torch.Size([546, 1, 28, 28]), torch.Size([546, 1]))

In [12]:
X = X.flatten(start_dim=1).cpu().numpy().T
Y = Y.cpu().numpy()
X.shape, Y.shape

((784, 546), (546, 1))

In [13]:
# Hyper Parameters
features = X.shape[0]
dim = 50
mu = 1
beta = 1
gamma = 1
sig2 = 0.5

In [14]:
# LDA's Sb and Sw
Sb, Sw, _,_ = calculate_L(X.T, Y)
St = Sb + Sw

Sb.shape, Sw.shape, St.shape

((784, 784), (784, 784), (784, 784))

In [15]:
# arbitrary columnly orthogonal matrix
# Random orthogonal matrix P projects data into a lower-dimensional space.
P = np.random.rand(features, dim)
P, _ = np.linalg.qr(P)  # QR decomposition for orthogonality
P.shape

(784, 50)

In [16]:
# Diagonal matrix D is initialized to penalize rows of P (related to L2,1-norm regularization).
norm_P_rows = np.linalg.norm(P, axis=1)  # Calculate L2 norm of rows
D = np.diag(1 / (norm_P_rows + 1e-12)) # Create diagonal matrix
D.shape

(784, 784)

In [17]:
# get first LS-SVM model in the initial P subspace.
p_train = P.T @ X;
p_train.shape # (features, n)

(50, 546)

In [18]:
Y.shape

(546, 1)

In [19]:
# Train an LS-SVM model on the projected training data
from lssvm import LSSVC
lssvc = LSSVC(gamma=gamma, kernel='linear')
lssvc.fit(p_train.T, Y)

In [20]:
alpha = lssvc.alpha.reshape(-1, 1)
alpha.shape

(546, 1)

In [21]:
print(p_train.shape, Y.shape, alpha.shape)
tmpv = X @ (alpha * Y)
tmpv.shape

(50, 546) (546, 1) (546, 1)


(784, 1)

In [22]:
TSum = tmpv @ tmpv.T
TSum.shape

(784, 784)

In [23]:
TSum.shape, Sw.shape, D.shape

((784, 784), (784, 784), (784, 784))

In [24]:
combined_matrix = TSum + beta * Sw + mu * D
trace_term = np.trace(P.T @ combined_matrix @ P)
alpha_norm_squared = np.sum(alpha ** 2)  # 等价于 ||alpha||_2^2
regularization_term = alpha_norm_squared / (2 * gamma)

In [25]:
obj = trace_term + regularization_term

In [26]:
for i in range(10):

    for j in range(10):
        # Sp
        Sp = TSum + mu * D + beta * Sw

        # lambda_n
        lambda_n = np.trace(P.T@St@P) / np.trace(P.T@Sp@P)
        B = St - lambda_n*Sp

        # Calculate the eigne vector
        B = (B + B.T) / 2
        Lambda, V = np.linalg.eig(B)
        # Sort eigenvalues in descending order and get the indices
        index = np.argsort(Lambda)[::-1]  # Get indices that would sort the eigenvalues
        P_old = P
        P = V[:, index[:dim]]  # Sort columns of eigenvectors

        # singular decomposition for the sake of orthogonal transformation invariance
        Sp_p = P @ P.T @ Sp @ P @ P.T
        tempU, _, _ = np.linalg.svd(Sp_p, full_matrices=False)  # Economy SVD
        P = tempU[:, :dim]
        D = np.eye(features)
        for ii in range(features):
            norm_value = np.linalg.norm(P[ii, :]) + 1e-12
            D[ii, ii] = 1.0 / norm_value

        if np.linalg.norm(P-P_old) < np.sqrt(features*dim)*0.01:
            print("inside loop converges")
            break

    p_train = P.T @ X
    lssvc = LSSVC(gamma=gamma, kernel='linear')
    lssvc.fit(p_train.T, Y)
    alpha = lssvc.alpha.reshape(-1, 1)
    # update TSum
    tmpv = X @ (alpha * Y)
    TSum = tmpv @ tmpv.T

    obj_old = obj
    obj = np.trace(P.T @ (TSum + beta*Sw + mu*D) @ P) + np.linalg.norm(alpha,ord=2) / (2*gamma)
    if abs(obj - obj_old)/abs(obj) <= 0.1:
        print(f"outside loop converges at iter {i}")
        break
    


inside loop converges
inside loop converges
inside loop converges
outside loop converges at iter 2


In [27]:
p_train = P.T @ X
p_test = P.T @ X_test
lssvc = LSSVC(gamma=gamma, kernel='linear')
lssvc.fit(p_train.T, Y)

In [28]:
# test 
help(lssvc.predict)

Help on method predict in module lssvm.LSSVC:

predict(X) method of lssvm.LSSVC.LSSVC instance
    Predicts the labels of data X given a trained model.
    - X: ndarray of shape (n_samples, n_attributes)



In [29]:
test_pred = lssvc.predict(p_test.T)
test_pred.shape,Y_test.shape

((156,), (156, 1))

In [30]:
100*np.sum(test_pred==Y_test[:, 0])/test_bs

73.71794871794872